# Quarter-five spot: point wells, layered media and a square obstacle

[Harder et al.](https://doi.org/10.1016/j.jcp.2013.03.019) study homogeneous and layered media with corner wells. Here the signed point functional is explicitly normalized to `v(1,1)-v(0,0)`. Its pressure is singular; comparisons away from wells do not assert a finite H1 energy. The separate square-obstacle benchmark below uses fixed finite-area wells.


The main equations execute from source and state their current discretization. The original archived comparisons below execute with `--historical` and require their exact attributed payloads. `--study` runs the complete public current acquisition declared in the catalogue, with its own output provenance.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/ea529866c28a9c6795b15b30f94347eef08ab3d540656dc68fc6ffebd38ba0df/22_quarter_five_spot-companion.zip"
COMPANION_SHA256 = "ea529866c28a9c6795b15b30f94347eef08ab3d540656dc68fc6ffebd38ba0df"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/22_quarter_five_spot.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


## Declare the local and global equations

The application writes local A, B and C blocks with the shared FEM kernels,
then supplies them to the generic assembler. Primal coefficients satisfy

$$
\begin{aligned}
a_T(p,v)&=(K\nabla p,\nabla v)_T,\\
L_T(v)&=(f,v)_T+\sum_j\alpha_jv(x_j),\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\\
C_Tp&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

Signs follow the fixed macroface normals. RT0 instead uses flux, pressure and
private boundary-pressure coefficients, including the local normal-flux
constraint. The global `Equation` supplies the signed Dirichlet moments and
retains each constant pressure mode. Pure Neumann data add the physical pressure
integral through an explicit constraint. `define_darcy` returns equations;
`assemble`, the global solve and field interpretation are separate stages.


In [ ]:
from pymhm import assemble, SolverConfig
from examples.formulations.darcy import define_darcy, pressure_constraints, recover_darcy

from pathlib import Path
import json
import sys
import numpy as np
from IPython.display import Image, display

from pymhm import TriangleMesh
from examples.plot_quarter_spot import WELLS, series_reference

mesh = TriangleMesh.unit_square(4)
for method in ("primal", "mixed"):
    solution_definition = define_darcy(
        mesh,
        point_sources=WELLS,
        formulation=method,
        degree=2 if method == "primal" else 1,
        local_refinement=2,
        neumann={int(f): 0 for f in mesh.boundary_faces},
    )
    solution_system = assemble(solution_definition.problem)
    solution = recover_darcy(
        solution_definition,
        solution_system,
        solution_system.solve(
            constraints=pressure_constraints(solution_definition, solution_system)
        ),
    )
    assert np.max(abs(solution.conservation_residuals())) < 1e-12
    assert sum((part[:, 2].sum() for part in solution.point_sources)) == 0
    if method == "mixed":
        assert max((np.max(abs(r)) for r in solution.fine_conservation_residuals())) < 1e-12
    print(method, np.max(abs(solution.conservation_residuals())))
points = np.array([[0.2, 0.3], [0.8, 0.7], [0.5, 0.5]])
p, q = series_reference(points)
assert abs(p[0] + p[1]) < 1e-13 and abs(p[2]) < 1e-13
np.testing.assert_allclose(q[0], q[1], atol=1e-13, rtol=0)
print("Independent Green-series values:", p, q)


The larger archived calculation has 2048 macrotriangles, two local subdivisions, constant traces and either primal P2 or mixed RT0/P0. The offset layer at $y=0.484375$ cuts macrocells but aligns with the fine mesh. Red lines mark material interfaces; dark lines mark macrofaces.

Figure 19 of Harder–Paredes–Valentin uses this offset interface, $\ell=0$, P2 and RT0 local solves, and the same numbers of macro and local edge subdivisions. The article does not specify the historical corner-load allocation, well intensity, exact connectivity or sampling of its elevation plot. The archived fields below use unit well rates and a stated diagonal connectivity. Their elevation views retain the calculated flux magnitudes without amplitude rescaling; singular peak values alone are not an accuracy criterion.

The full elevation uses a common scale with the actual nodal display peaks. The detail view excludes only display triangles intersecting the open disk of radius 0.125 around either well; it leaves the numerical solution and well rates unchanged. Each view uses the same camera and height scale for P2 and RT0.

An independently assembled Basix 0.9.0 comparison verifies all six discrete cases in the same spaces, geometry, point allocations and physical gauge. The published record binds each candidate archive to the executed reference bases and original physical equations. This establishes discrete agreement; it does not identify the original paper's missing inputs or establish continuum accuracy. The native reference uses a fixed operator and uniform extended residual accumulation; its field stability is checked by a further correction of the same equations.


In [ ]:
if HISTORICAL_REPLAY:
    import hashlib
    from examples.quarter_point_archive import replay_point_fields
    from threadpoolctl import threadpool_limits

    point_folder = ROOT / "examples/results/quarter-five-spot"
    point_record = json.loads((point_folder / "point-wells.json").read_text())
    assert len(point_record["rows"]) == 6
    for row in point_record["rows"]:
        field_path = point_folder / row["field_archive"]
        assert hashlib.sha256(field_path.read_bytes()).hexdigest() == row["field_archive_sha256"]
        metadata = json.loads(field_path.with_suffix(".json").read_text())
        assert metadata["archive_sha256"] == row["field_archive_sha256"]
        assert metadata["macro_cells"] == 2048 and metadata["local_refinement"] == 2
        assert metadata["original_saddle_relative_load_residual"] <= 1e-10
        assert abs(metadata["physical_pressure_integral"]) <= 1e-10
        with np.load(field_path) as archive:
            fields = dict(archive)
        for threads in (1, 2):
            with threadpool_limits(threads):
                pressure, flux = replay_point_fields(fields, metadata)
            np.testing.assert_allclose(pressure, fields["pressure_quadrature"], rtol=2e-14, atol=2e-14)
            np.testing.assert_allclose(flux, fields["flux_quadrature"], rtol=2e-14, atol=2e-14)
        print(row["name"], row["formulation"], "original saddle residual",
              metadata["original_saddle_relative_load_residual"])
        del fields, pressure, flux

    native = json.loads((point_folder / "reference/point-native-verification.json").read_text())
    assert native["candidate_point_record_sha256"] == hashlib.sha256((point_folder / "point-wells.json").read_bytes()).hexdigest()
    assert len(native["rows"]) == 6 and native["geometry"]["macro_triangles"] == 2048
    point_rows = {(row["name"], row["formulation"]): row for row in point_record["rows"]}
    for row in native["rows"]:
        original = point_rows[row["name"], row["formulation"]]
        assert row["candidate_field_archive_sha256"] == original["field_archive_sha256"]
        assert row["native_original_physical_relative_residual"] <= native["native_arithmetic"]["original_physical_relative_residual_criterion"]
        assert row["candidate_in_native_original_physical_relative_residual"] <= 1e-10
        assert abs(row["native_physical_pressure_integral"]) <= 1e-10
        assert row["pressure_relative_l2_difference"] <= native["norms"]["field_agreement_criterion"]
        assert row["flux_relative_l2_difference"] <= native["norms"]["field_agreement_criterion"]
        assert all(row["one_two_threads_replay"].values())
        print("Independent Basix", row["name"], row["formulation"],
              "relative pressure/flux differences", row["pressure_relative_l2_difference"],
              row["flux_relative_l2_difference"])

    display(Image(filename=str(ROOT / "docs/figures/quarter-five-spot/point-flux-elevation.png")))
    record = json.loads((point_folder / "point-convergence.json").read_text())
    assert len(record["rows"]) == 6
    assert record["rows"][-1]["sampled_pressure_rms"] < record["rows"][0]["sampled_pressure_rms"]
    for row in record["rows"]:
        assert hashlib.sha256((point_folder / row["archive"]).read_bytes()).hexdigest() == row["archive_sha256"]
        assert row["original_saddle_relative_load_residual"] <= 1e-10
        print(row["macro_triangles"], row["sampled_pressure_rms"], row["sampled_flux_rms"])


## A square obstacle with interfaces inside macrocells

Define $a=0.25$, $b=0.75$ and $D=[a,b]^2$. The centered obstacle has side length $0.5$ and area $0.25$, or 25% of the domain, with $K=10^{-4}$ in $D$ and $K=1$ outside. The equations are $q=-K\nabla p$ and $\nabla\cdot q=f$, with zero exterior normal flux and zero domain-mean pressure. The source is $-100$ on $[0,0.1]^2$ and $+100$ on $[0.9,1]^2$, so the fixed well rates are $-1$ and $+1$.

The macro mesh splits a uniform $10\times10$ square grid along southwest-to-northeast diagonals, giving 200 macrotriangles. The obstacle interface cuts macrocells; it is resolved exactly by the local meshes with 4, 8 or 16 subdivisions per macroedge. This tests heterogeneous local problems without averaging permeability within a macrocell. It does not test an interface cutting the fine elements. The fixed finite-area wells define a separate regularized problem, distinct from the published Dirac-well experiment.


## Matched Basix and NeoPZ obstacle comparisons

The eight configurations per reference combine local subdivisions $r\in\{4,8,16\}$ and one or two constant trace segments, with four segments also tested at $r=8,16$. Basix tabulates the basis and quadrature for an independently assembled full primal P1 saddle. NeoPZ supplies a native RT0 operator with explicit macro trace restrictions and a physical mean-pressure gauge. Private instrumented drivers execute these comparisons; neither comparison is an original author application.

The component plots retain the signs of $q_x$ and $q_y$, with symmetric color scales and a common range for each reference/PyMHM pair. The vector arrows display $q/|q|$ at actual fine-cell centroids selected by spatial bins. Their fixed length of 0.035 in unit-domain coordinates encodes direction only; the common background color scale shows magnitude. Both implementations use the same selected centroids, with no averaging across interfaces. Difference maps distinguish signed component differences from the norm of the vector difference.

The primal local P1 pressure is continuous, but its raw flux $-K\nabla p_h$ need not have continuous normal components across fine faces. The RT0/P0 local pair satisfies $\nabla\cdot RT0=P0$; normal-flux conformity and fine-cell balance follow from its compatible mixed spaces and macro trace restrictions ([Durán et al., §§3.2–3.3](https://doi.org/10.1016/j.cma.2019.05.013)). A jump in the tangential flux is physically admissible at the material interface.

The interface intersects macrofaces at their midpoints. Two trace segments place an endpoint at each material intersection; one segment does not. This geometric condition is relevant to the analysis in [Chaumont-Frelet, Paredes and Valentin (2026)](https://doi.org/10.1016/j.camwa.2026.01.016), whose skeletal subfaces each lie within one material region. Matching this geometry alone does not verify the paper's convergence estimates or eliminate the error of a fixed trace space.


In [ ]:
if HISTORICAL_REPLAY:
    import hashlib

    from examples.quarter_spot_problem import (
        OBSTACLE_AREA, OBSTACLE_LOWER, OBSTACLE_UPPER,
        coefficient, macro_mesh, source,
    )

    obstacle_mesh = macro_mesh()
    assert len(obstacle_mesh.cells) == 200
    np.testing.assert_array_equal(obstacle_mesh.points, TriangleMesh.unit_square(10).points)
    np.testing.assert_allclose([OBSTACLE_LOWER, OBSTACLE_UPPER], [0.25, 0.75], atol=1e-15, rtol=0)
    np.testing.assert_allclose((OBSTACLE_UPPER - OBSTACLE_LOWER)**2, 0.25, atol=1e-15, rtol=0)
    # Material interfaces lie inside macrocells, rather than on macro grid lines.
    for position in (OBSTACLE_LOWER, OBSTACLE_UPPER):
        assert np.min(abs(obstacle_mesh.points[:, 0] - position)) > 0.02
    # The coarsest local mesh already fits those interfaces; integrate material
    # and well areas on that fine mesh, not by one sample per macrocell.
    fine_meshes = [obstacle_mesh.submesh(cell, 4) for cell in range(len(obstacle_mesh.cells))]
    triangles = np.concatenate([fine.points[fine.cells] for fine in fine_meshes])
    areas = np.concatenate([fine.areas for fine in fine_meshes])
    centroids = triangles.mean(axis=1)
    material = coefficient(centroids)
    material_by_macro = material.reshape(len(obstacle_mesh.cells), 16)
    cut_macros = np.any(material_by_macro == 1e-4, axis=1) & np.any(material_by_macro == 1, axis=1)
    assert np.count_nonzero(cut_macros) == 38
    well_density = source(centroids)
    for position in (OBSTACLE_LOWER, OBSTACLE_UPPER):
        crossed = (triangles.min(axis=1) < position - 1e-14) & (triangles.max(axis=1) > position + 1e-14)
        assert not np.any(crossed)
    np.testing.assert_allclose(areas[material == 1e-4].sum(), OBSTACLE_AREA, atol=1e-15, rtol=0)
    np.testing.assert_allclose(np.sum(areas * np.maximum(well_density, 0)), 1.0, atol=1e-15, rtol=0)
    np.testing.assert_allclose(np.sum(areas * np.minimum(well_density, 0)), -1.0, atol=1e-15, rtol=0)

    archive = ROOT / "examples/results/quarter-five-spot/reference"
    comparison = json.loads((archive / "comparison.json").read_text())
    assert comparison["obstacle_area"] == OBSTACLE_AREA
    assert len(comparison["cases"]) == 16
    assert {row["refinement"] for row in comparison["cases"]} == {4, 8, 16}
    for row in comparison["cases"]:
        assert hashlib.sha256((archive / row["fields"]).read_bytes()).hexdigest() == row["fields_sha256"]
        with np.load(archive / row["fields"]) as fields:
            np.testing.assert_allclose(fields["macro_points"], obstacle_mesh.points, atol=1e-15, rtol=0)
            np.testing.assert_array_equal(fields["macro_cells"], obstacle_mesh.cells)
    assert max(r["pressure_l2_difference"] for r in comparison["cases"]) < 1e-10
    assert max(r["flux_l2_difference"] for r in comparison["cases"]) < 1e-10
    print("Largest pressure L2 difference:", max(r["pressure_l2_difference"] for r in comparison["cases"]))
    print("Largest flux L2 difference:", max(r["flux_l2_difference"] for r in comparison["cases"]))
    display(Image(filename=str(ROOT / "docs/figures/quarter-five-spot/obstacle-macro-cut.png")))


## Classical RT0 reference and trace restriction

The classical NeoPZ RT0/P0 solution retains independent normal fluxes on every fine face, with zero exterior normal flux and mean-zero pressure. Successive material-aligned refinements are recorded in the table below. Successive pressure and flux differences are integrated on the finer triangles and normalized by the finer field norm; they measure refinement sensitivity rather than exact-solution error.

The MHM comparison holds $r=16$ fixed and uses one, two or four constant segments per macroface. Its same-grid differences use the classical $r=16$ field norm as denominator. Separate comparisons with the finest available classical field include local and skeletal approximation. The one-segment trace spans each material intersection, while two segments resolve the intersection at the midpoint. Four segments further enrich the trace approximation. The last classical refinement increment is a comparison scale, not an error bound.

The measured refinement increments and MHM differences are printed from the digested current archives below. They are numerical-field comparisons, not certified errors against the exact PDE solution. All three trace segment counts at $r=16$ are included in the full native NeoPZ comparisons.


In [ ]:
if HISTORICAL_REPLAY:
    import hashlib

    classical = json.loads((archive / "classical-convergence.json").read_text())
    assert classical["geometry"]["obstacle_area"] == OBSTACLE_AREA
    levels = [level["refinement"] for level in classical["levels"]]
    assert len(levels) >= 3 and all(b == 2*a for a,b in zip(levels[:-1], levels[1:]))
    assert {row["segments"] for row in classical["mhm_trace_enrichment"]} == {1, 2, 4}
    for row in classical["levels"] + classical["mhm_trace_enrichment"]:
        assert hashlib.sha256((archive / row["fields"]).read_bytes()).hexdigest() == row["fields_sha256"]
    for row in classical["levels"]:
        assert row["algebra"]["native_free_original_relative_load_residual"] < 1e-10
        with np.load(archive / row["fields"]) as fields:
            fine = TriangleMesh(fields["points"], fields["cells"])
            flux_squared = np.sum(fields["reference_flux_quadrature"]**2, axis=2) @ fields["quadrature_weights"]
            energy = np.sum(fine.areas * flux_squared / fields["permeability"])
            work = np.sum(fine.areas * fields["reference_pressure"] * fields["source_density"])
            relative_defect = abs(energy - work) / max(abs(energy), abs(work))
            assert relative_defect < 1e-11
            print(f"Classical r={row['refinement']}: energy identity relative defect {relative_defect:.3e}")
    for row in classical["refinement_differences"]:
        print(f"Classical r={row['coarse_refinement']} to {row['fine_refinement']}: "
              f"pressure change {100*row['pressure_l2_relative']:.6f}%, "
              f"flux change {100*row['flux_l2_relative']:.6f}%")
    for row in classical["mhm_trace_enrichment"]:
        print(f"MHM r={row['refinement']}, segments={row['segments']}: "
              f"pressure difference {100*row['pressure_l2_relative']:.6f}%, "
              f"flux difference {100*row['flux_l2_relative']:.6f}%")

    display(Image(filename=str(ROOT / "docs/figures/quarter-five-spot/classical-vs-mhm-components.png")))
